# Picotron demo: SmolLM2-135M → SFT → DPO

This Kaggle-ready demo proves that Picotron's script-first SFT and DPO layers can train a real Hugging Face causal LM, not only Picotron's toy model. It uses bounded subsets and update counts for a practical demo runtime.


## 1. Clone Picotron and install dependencies


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/kaggle/working/picotron')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/AndyMLAndAI/picotron.git', str(REPO)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'accelerate'], check=True)


## 2. Load the real SmolLM2-135M checkpoint

`HuggingFaceTB/SmolLM2-135M` is loaded through `transformers.AutoModelForCausalLM`. The runtime assertions document the actual architecture rather than assuming it: Llama-style decoder, 576 hidden size, 30 layers, 9 query heads, 3 KV heads (GQA), and a 49,152-token vocabulary. FP32 is intentional here: Picotron's generic SFT/DPO scripting loops do not yet implement mixed-precision gradient scaling for arbitrary HF models.


In [ ]:
import itertools
from dataclasses import replace

import torch
from datasets import load_dataset
from torch.utils.data import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from picotron.config.config import load_config
from picotron_sft import run_sft
from picotron_dpo import run_dpo

MODEL_ID = 'HuggingFaceTB/SmolLM2-135M'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'Run this GPU demo with Kaggle GPU acceleration enabled.'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(DEVICE)
model.config.use_cache = False  # training does not need a KV cache

hf_config = model.config
print({name: getattr(hf_config, name, 'N/A') for name in (
    'model_type', 'hidden_size', 'intermediate_size', 'num_hidden_layers',
    'num_attention_heads', 'num_key_value_heads', 'vocab_size', 'rope_theta',
)})
assert hf_config.model_type == 'llama'
assert (hf_config.hidden_size, hf_config.num_hidden_layers, hf_config.vocab_size) == (576, 30, 49152)

# TrainingDisplay needs a PicotronConfig only for its banner and cadence.
base_display_config = load_config(REPO / 'src/picotron/config/toy_model.yaml')
display_config = replace(
    base_display_config,
    model=replace(base_display_config.model, model_config=replace(
        base_display_config.model.model_config,
        vocab_size=hf_config.vocab_size,
        hidden_size=hf_config.hidden_size,
        intermediate_size=hf_config.intermediate_size,
        num_hidden_layers=hf_config.num_hidden_layers,
        num_attention_heads=hf_config.num_attention_heads,
        num_key_value_heads=hf_config.num_key_value_heads,
    )),
    data=replace(base_display_config.data, vocab_size=hf_config.vocab_size),
    tokens=replace(base_display_config.tokens, sequence_length=256, micro_batch_size=2),
    optimizer=replace(base_display_config.optimizer, learning_rate_scheduler=replace(
        base_display_config.optimizer.learning_rate_scheduler, learning_rate=2e-5,
    )),
)


## 3. Baseline inference


In [ ]:
PROMPTS = [
    'Explain why the sky appears blue in two sentences.',
    'Write a short Python function that checks whether a number is prime.',
]

def generate_all(active_model):
    active_model.eval()
    previous_use_cache = active_model.config.use_cache
    active_model.config.use_cache = True
    outputs = []
    with torch.no_grad():
        for prompt in PROMPTS:
            encoded = tokenizer(prompt, return_tensors='pt').to(DEVICE)
            generated = active_model.generate(
                **encoded, max_new_tokens=96, do_sample=False,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
            )
            outputs.append(tokenizer.decode(generated[0], skip_special_tokens=True))
    active_model.config.use_cache = previous_use_cache
    return outputs

base_outputs = generate_all(model)
for prompt, output in zip(PROMPTS, base_outputs):
    print(f'PROMPT: {prompt}\nBASE: {output}\n')


## 4. Supervised fine-tuning on SmolTalk

We stream and materialize 5,000 conversations from the `all`/`train` split, then run 500 update steps (batch size 2) to bound demo time. This is deliberately a smoke-scale adaptation, not a full reproduction of SmolLM2-Instruct training. Labels include the full rendered conversation; assistant-only loss masking is not claimed.


In [ ]:
MAX_LENGTH = 256
SFT_EXAMPLES = 5_000
SFT_STEPS = 500

smoltalk_stream = load_dataset('HuggingFaceTB/smoltalk', 'all', split='train', streaming=True)
smoltalk_examples = list(itertools.islice(smoltalk_stream, SFT_EXAMPLES))
assert len(smoltalk_examples) == SFT_EXAMPLES

class SmolTalkSFTDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        messages = self.examples[index]['messages']
        if getattr(tokenizer, 'chat_template', None):
            input_ids = tokenizer.apply_chat_template(
                messages, tokenize=True, add_generation_prompt=False,
            )
        else:
            text = '\n'.join(f"{item['role']}: {item['content']}" for item in messages)
            input_ids = tokenizer.encode(text, add_special_tokens=False)
        input_ids = input_ids[:MAX_LENGTH]
        padding = MAX_LENGTH - len(input_ids)
        return {
            'input_ids': torch.tensor(input_ids + [tokenizer.pad_token_id] * padding, dtype=torch.long),
            'attention_mask': torch.tensor([1] * len(input_ids) + [0] * padding, dtype=torch.long),
            'labels': torch.tensor(input_ids + [-100] * padding, dtype=torch.long),
        }

sft_losses = run_sft(
    model=model,
    dataset=SmolTalkSFTDataset(smoltalk_examples),
    learning_rate=2e-5,
    batch_size=display_config.tokens.micro_batch_size,
    num_steps=SFT_STEPS,
    device=DEVICE,
    display_config=display_config,
)
SFT_DIR = REPO / 'artifacts' / 'smollm2_135m_smoltalk_sft'
SFT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SFT_DIR)
tokenizer.save_pretrained(SFT_DIR)
print(f'SFT complete: first_loss={sft_losses[0]:.4f}, last_loss={sft_losses[-1]:.4f}')


## 5. Inference after SFT


In [ ]:
sft_outputs = generate_all(model)
for prompt, output in zip(PROMPTS, sft_outputs):
    print(f'PROMPT: {prompt}\nSFT: {output}\n')


## 6. DPO on UltraFeedback Binarized

We use 2,000 streamed examples from `train_prefs`, but 300 update steps for a bounded demo. The current UltraFeedback schema stores `chosen` and `rejected` as message lists, so the cell extracts the final assistant completion from each and uses the model chat template for the shared prompt. `run_dpo` automatically snapshots the SFT model as its frozen reference before policy updates. Its Rich table reports DPO loss, chosen/rejected log-probabilities, and margin.


In [ ]:
DPO_EXAMPLES = 2_000
DPO_STEPS = 300

def last_assistant_content(messages):
    for message in reversed(messages):
        if message.get('role') == 'assistant' and isinstance(message.get('content'), str):
            return message['content']
    return None

def dpo_prompt(prompt):
    messages = [{'role': 'user', 'content': prompt}]
    if getattr(tokenizer, 'chat_template', None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return f'user: {prompt}\nassistant: '

preference_stream = load_dataset(
    'HuggingFaceH4/ultrafeedback_binarized', split='train_prefs', streaming=True,
)
dpo_triples = []
for example in preference_stream:
    chosen = last_assistant_content(example['chosen'])
    rejected = last_assistant_content(example['rejected'])
    if isinstance(example['prompt'], str) and chosen and rejected:
        dpo_triples.append((dpo_prompt(example['prompt']), chosen, rejected))
    if len(dpo_triples) == DPO_EXAMPLES:
        break
assert len(dpo_triples) == DPO_EXAMPLES

dpo_losses = run_dpo(
    model=model,  # already SFT-adapted; run_dpo snapshots this as the frozen reference
    dataset=dpo_triples,
    tokenizer=tokenizer,
    beta=0.1,
    learning_rate=1e-5,
    batch_size=1,
    max_length=MAX_LENGTH,
    num_steps=DPO_STEPS,
    device=DEVICE,
    display_config=display_config,
)
DPO_DIR = REPO / 'artifacts' / 'smollm2_135m_smoltalk_dpo'
DPO_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(DPO_DIR)
tokenizer.save_pretrained(DPO_DIR)
print(f'DPO complete: first_loss={dpo_losses[0]:.4f}, last_loss={dpo_losses[-1]:.4f}')


## 7. Compare base, SFT, and DPO generations

Qualitative changes from this tiny run may be subtle. The meaningful validation is that the same arbitrary HF model completed baseline inference, SFT, DPO, and post-DPO inference without architecture-specific Picotron code.


In [ ]:
dpo_outputs = generate_all(model)
for prompt, base, sft, dpo in zip(PROMPTS, base_outputs, sft_outputs, dpo_outputs):
    print('=' * 100)
    print(f'PROMPT: {prompt}')
    print(f'\nBASE:\n{base}')
    print(f'\nSFT:\n{sft}')
    print(f'\nDPO:\n{dpo}')

print('\nKaggle verification checklist:')
print('- Confirm model/dataset downloads, GPU memory, and complete SFT/DPO runs.')
print('- Confirm the displayed DPO chosen-vs-rejected margin trends upward.')
print('- This notebook was authored locally without GPU or HF-network execution.')
print('- Architecture note: SmolLM2 is Llama/GQA/RoPE, but HF handles those internals; Picotron only assumes input_ids/attention_mask -> logits.')
